In [9]:
import numpy as np
from case_studies import *

# https://en.wikipedia.org/wiki/Wolfe_conditions For c_1 and c_2 choice
#def wolfe_line_search(f, df, x, p, c1=1e-4, c2=0.9, max_iter=1000):
#    a = 1.0
#    fx = f(x)
#    grad_x = df(x)
#    for _ in range(max_iter):
#        new_x = x + a * p
#        if f(new_x) > fx + c1 * a * np.dot(p, grad_x):
#            a *= 0.5  # Reduce step size
#        elif np.dot(p, df(new_x)) < c2 * np.dot(p, grad_x):
#            a *= 2.0  # Increase step size
#        else:
#            break
#    return a

def is_pos_def(H):
    return np.all(np.linalg.eigvals(H) > 0)

def backtrack(f, df, x, p, alpha_init=1.0, rho=0.5, c1=1e-4):
    fx = f(x)
    grad_x = df(x)
    alpha = alpha_init
    while f(x + alpha * p) > fx + c1 * alpha * np.dot(grad_x, p):
        alpha *= rho
    return alpha

# Newton's Method function (updated version)
def newtons_method(f, df, Hf, x0, max_iter=10000, epsilon=1e-6):
    x = x0.copy()
    xs = [x]
    for _ in range(max_iter):
        if np.linalg.norm(df(x)) < epsilon: break
        H = Hf(x)
        if np.linalg.eigvals(H).min() > 0:  # Check if Hessian is positive definite
            p = -np.linalg.solve(H, df(x))
        else:
            eigvals, eigvecs = np.linalg.eigh(H)
            new_H = sum((1 / abs(l)) * np.outer(v, v) for l, v in zip(eigvals, eigvecs.T))
            p = -new_H @ df(x)
        alpha = backtrack(f, df, x, p, 1.0)
        x = x + alpha * p
        xs.append(x)
    return np.array(xs)

# Steepest Descent function (updated version)
def steepest_descent(f, df, x0, max_iter=10000, epsilon=1e-6):
    x = x0.copy()
    xs = [x]
    for _ in range(max_iter):
        if np.linalg.norm(df(x)) < epsilon: break
        p = -df(x)
        alpha = backtrack(f, df, x, p, 1.0)
        x = x + alpha * p
        xs.append(x)
    return np.array(xs)


# Newton's Method function (updated version)
def CG(Q, g, epsilon=1e-6):
    cgc = 0 #CG step count
    x_k = 0
    nabla_f_k = g
    p_k = -nabla_f_k
    for _ in range(len(g)):
        cgc += 1
        Qp_k = Q @ p_k
        a_k = -(np.transpose(p_k) @ nabla_f_k / (np.transpose(p_k) @ Qp_k))
        x_k = x_k + a_k * p_k
        nabla_f_k = Q @ x_k + g
        
        if np.linalg.norm(nabla_f_k) < epsilon: break

        upper = np.transpose(nabla_f_k.T) @ Qp_k
        lower = np.transpose(p_k @ Qp_k)
        
        p_k = -nabla_f_k + (upper/lower) * p_k
    
    return x_k, cgc

def newton_CG(f, df, Hf, x_0, c1=1e-4, rho=0.5, max_iter=1000, epsilon=1e-6):
    cgcl = [] #CG step count list
    x_k = x_0
    xs = [x_k]
    
    for _ in range(max_iter):
        nabla_f_k = df(x_k)
        if np.linalg.norm(nabla_f_k) < epsilon: break
        
        eta_k = 0.5 * min(0.5, np.sqrt(np.linalg.norm(nabla_f_k)))
        epsilon_k = eta_k * np.linalg.norm(nabla_f_k)
        
        Q_k = Hf(x_k)        
        p_k, cgc = CG(Q_k, nabla_f_k, epsilon_k)
        cgcl.append(cgc)
        a_k = backtrack(f, df, x_k, p_k, 1.0, rho, c1)
        x_k = x_k + a_k * p_k
        xs.append(x_k)
    
    return np.array(xs), np.array(cgcl)


x0_f1 = np.random.randn(5)
x0_f2 = np.random.randn(2)
x0_f3 = np.random.randn(5)
x0_f4 = np.random.randn(5)
x0_f5 = np.random.randn(5)

print("Steepest Descent Results:")
print("f1:", steepest_descent(f1, df1, x0_f1)[0])
print("f2:", steepest_descent(f2, df2, x0_f2)[0])
print("f3:", steepest_descent(f3, df3, x0_f3)[0])
print("f4:", steepest_descent(f4, df4, x0_f4)[0])
print("f5:", steepest_descent(f5, df5, x0_f5)[0])

# Running Newton's Method
print("\nNewton's Method Results:")
print("f1:", newtons_method(f1, df1, Hf1, x0_f1)[0])
print("f2:", newtons_method(f2, df2, Hf2, x0_f2)[0])
print("f3:", newtons_method(f3, df3, Hf3, x0_f3)[0])
print("f4:", newtons_method(f4, df4, Hf4, x0_f4)[0])
print("f5:", newtons_method(f5, df5, Hf5, x0_f5)[0])

# Running Newton_CG Method
print("\nNewton's Method Results:")
a, b = newton_CG(f1, df1, Hf1, x0_f1)
print("f1:", a[0], b)
a, b = newton_CG(f2, df2, Hf2, x0_f2)
print("f2:", a[0], b)
a, b = newton_CG(f3, df3, Hf3, x0_f3)
print("f3:", a[0], b)
a, b = newton_CG(f4, df4, Hf4, x0_f4)
print("f4:", a[0], b)
a, b = newton_CG(f5, df5, Hf5, x0_f5)
print("f5:", a[0], b)

Steepest Descent Results:
f1: [ 0.85924054  0.85359439 -0.7070327  -2.40609649 -0.93212881]
f2: [-0.13337082  2.6746802 ]
f3: [-0.49416276 -0.05613582  0.13038751  0.15326771  0.9898455 ]
f4: [-0.74847408 -0.53660982  0.23363384 -0.5441983  -0.19536534]
f5: [-0.77625192 -0.49380706  0.07980362 -0.89600836  0.01181499]

Newton's Method Results:
f1: [ 0.85924054  0.85359439 -0.7070327  -2.40609649 -0.93212881]
f2: [-0.13337082  2.6746802 ]
f3: [-0.49416276 -0.05613582  0.13038751  0.15326771  0.9898455 ]
f4: [-0.74847408 -0.53660982  0.23363384 -0.5441983  -0.19536534]
f5: [-0.77625192 -0.49380706  0.07980362 -0.89600836  0.01181499]

Newton's Method Results:
f1: [ 0.85924054  0.85359439 -0.7070327  -2.40609649 -0.93212881] [2 4 5]
f2: [-0.13337082  2.6746802 ] [2 2 2 1 2 1 2 2 1 2 1 2 1 2 1 2 1 2 1 2 1]
f3: [-0.49416276 -0.05613582  0.13038751  0.15326771  0.9898455 ] [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 